# Toy Brick Optimization Model (Large Scale) - Snowflake Edition

This notebook demonstrates large-scale optimization using Snowflake's capabilities including:
- User-Defined Functions (UDFs) for optimization
- Parallel processing
- Table functions for scalable optimization

## Prerequisites
- Completed notebooks 01 and 02
- Gurobi license appropriate for larger models
- Snowflake warehouse with adequate compute

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, lit, sum as sum_, udf, udtf
from snowflake.snowpark.types import *
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import json

# Configuration
DATABASE = "TOY_BRICK_DB"
SCHEMA = "RAW_DATA"

try:
    session = snowpark.Session.builder.getOrCreate()
except:
    connection_parameters = {
        "account": "your_account",
        "user": "your_user",
        "password": "your_password",
        "role": "your_role",
        "warehouse": "your_warehouse",
        "database": DATABASE,
        "schema": SCHEMA
    }
    session = Session.builder.configs(connection_parameters).create()

print(f"Connected to: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

## Step 1: Create Large-Scale Problem

Instead of just 4 sets, let's consider a collector who has accumulated 100+ sets over the years.

In [ ]:
# Select a large collection of owned sets
# For demonstration, we'll select sets from popular themes

owned_sets_large = session.sql("""
SELECT set_num
FROM sets s
JOIN themes t ON s.theme_id = t.id
WHERE t.name IN ('City', 'Creator', 'Friends', 'Technic')
  AND s.year BETWEEN 2015 AND 2020
  AND s.num_parts BETWEEN 50 AND 500
ORDER BY RANDOM()
LIMIT 100
""").to_pandas()

owned_set_list = owned_sets_large['SET_NUM'].tolist()
print(f"Large collection: {len(owned_set_list)} sets")

# Calculate total inventory
owned_sets_str = "','".join(owned_set_list)

session.sql(f"""
CREATE OR REPLACE TABLE large_inventory AS
SELECT 
    part_color_id,
    part_num,
    color_id,
    part_name,
    color_name,
    SUM(quantity) as available_quantity
FROM v_set_parts
WHERE set_num IN ('{owned_sets_str}')
GROUP BY part_color_id, part_num, color_id, part_name, color_name
""").collect()

# Get statistics
stats = session.sql("""
SELECT 
    COUNT(*) as unique_part_colors,
    SUM(available_quantity) as total_pieces,
    AVG(available_quantity) as avg_per_part_color,
    MAX(available_quantity) as max_quantity
FROM large_inventory
""").collect()[0]

print(f"\nInventory Statistics:")
print(f"  Unique part-color combinations: {stats['UNIQUE_PART_COLORS']:,}")
print(f"  Total pieces: {stats['TOTAL_PIECES']:,}")
print(f"  Average per part-color: {stats['AVG_PER_PART_COLOR']:.1f}")
print(f"  Max quantity of any part-color: {stats['MAX_QUANTITY']:,}")

## Step 2: Create Candidate Sets Pool

For large-scale optimization, we'll consider many more candidate sets.

In [ ]:
# Get a large pool of candidate sets
session.sql(f"""
CREATE OR REPLACE TABLE candidate_sets_large AS
SELECT DISTINCT
    s.set_num,
    s.name as set_name,
    s.year,
    s.theme_id,
    t.name as theme_name,
    s.num_parts
FROM sets s
JOIN themes t ON s.theme_id = t.id
WHERE s.year BETWEEN 2015 AND 2022
  AND s.num_parts < 1000
  AND s.theme_id IN (
      SELECT DISTINCT theme_id 
      FROM sets 
      WHERE set_num IN ('{owned_sets_str}'))
ORDER BY s.num_parts DESC
LIMIT 500
""").collect()

candidate_count = session.sql("SELECT COUNT(*) as cnt FROM candidate_sets_large").collect()[0]['CNT']
print(f"Candidate sets pool: {candidate_count:,}")

# Show distribution by theme
print("\nCandidate sets by theme:")
session.sql("""
SELECT theme_name, COUNT(*) as set_count, 
       AVG(num_parts) as avg_parts,
       SUM(num_parts) as total_parts
FROM candidate_sets_large
GROUP BY theme_name
ORDER BY set_count DESC
""").show()

## Step 3: Create Optimization UDF

We'll create a Snowflake UDF that can run optimization in parallel across different scenarios.

In [ ]:
# Define optimization function that will become a UDF
def optimize_brick_selection(inventory_json: str, requirements_json: str, 
                              objective_type: str, max_sets: int) -> str:
    """
    Optimize toy brick set selection.
    
    Parameters:
    - inventory_json: JSON string of available inventory {part_color_id: quantity}
    - requirements_json: JSON string of set requirements {set_num: {part_color_id: quantity}}
    - objective_type: 'maximize_parts' or 'maximize_sets'
    - max_sets: Maximum number of sets to consider
    
    Returns:
    - JSON string with solution details
    """
    import gurobipy as gp
    from gurobipy import GRB
    import json
    
    try:
        # Parse inputs
        inventory = json.loads(inventory_json)
        requirements = json.loads(requirements_json)
        
        # Create model
        model = gp.Model("ToyBrickOpt")
        model.setParam('OutputFlag', 0)
        model.setParam('TimeLimit', 300)  # 5 minute time limit
        
        # Decision variables
        set_vars = {}
        for set_num in requirements.keys():
            set_vars[set_num] = model.addVar(vtype=GRB.BINARY, name=f"build_{set_num}")
        
        # Objective
        if objective_type == 'maximize_parts':
            obj_expr = gp.LinExpr()
            for set_num, parts_needed in requirements.items():
                total_parts = sum(parts_needed.values())
                obj_expr += total_parts * set_vars[set_num]
        else:  # maximize_sets
            obj_expr = gp.quicksum(set_vars.values())
        
        model.setObjective(obj_expr, GRB.MAXIMIZE)
        
        # Constraints: inventory limits
        for part_color_id, available in inventory.items():
            constraint_expr = gp.LinExpr()
            for set_num, parts_needed in requirements.items():
                if part_color_id in parts_needed:
                    constraint_expr += parts_needed[part_color_id] * set_vars[set_num]
            
            if constraint_expr.size() > 0:
                model.addConstr(constraint_expr <= available)
        
        # Optional: limit number of sets
        if max_sets > 0:
            model.addConstr(gp.quicksum(set_vars.values()) <= max_sets)
        
        # Solve
        model.optimize()
        
        # Extract solution
        if model.status == GRB.OPTIMAL:
            selected_sets = [s for s, v in set_vars.items() if v.X > 0.5]
            
            result = {
                'status': 'optimal',
                'selected_sets': selected_sets,
                'num_sets': len(selected_sets),
                'objective_value': model.ObjVal,
                'solve_time': model.Runtime
            }
        else:
            result = {
                'status': 'infeasible' if model.status == GRB.INFEASIBLE else 'other',
                'selected_sets': [],
                'num_sets': 0,
                'objective_value': 0,
                'solve_time': 0
            }
        
        return json.dumps(result)
        
    except Exception as e:
        return json.dumps({
            'status': 'error',
            'error': str(e),
            'selected_sets': [],
            'num_sets': 0,
            'objective_value': 0
        })

# Register UDF
# Note: This requires gurobipy to be available in Snowflake
# You may need to upload it as a package

optimize_udf = udf(
    optimize_brick_selection,
    return_type=StringType(),
    input_types=[StringType(), StringType(), StringType(), IntegerType()],
    packages=['gurobipy', 'pandas'],
    is_permanent=False,
    name='optimize_brick_selection',
    replace=True
)

print("UDF created: optimize_brick_selection")

## Step 4: Prepare Data for Batch Optimization

We'll create different scenarios to test.

In [ ]:
# For large-scale problem, we'll use Python for optimization
# (UDF approach requires Gurobi in Snowflake environment which needs special setup)

print("Preparing large-scale optimization...")

# Get inventory as dictionary
inventory_df = session.sql("SELECT part_color_id, available_quantity FROM large_inventory").to_pandas()
inventory_dict = dict(zip(inventory_df['PART_COLOR_ID'], inventory_df['AVAILABLE_QUANTITY']))

# Get candidate sets
candidate_sets_df = session.sql("SELECT * FROM candidate_sets_large").to_pandas()
candidate_list = candidate_sets_df['SET_NUM'].tolist()

# Get requirements
candidate_sets_str = "','".join(candidate_list)
requirements_df = session.sql(f"""
SELECT set_num, part_color_id, quantity
FROM set_part_requirements
WHERE set_num IN ('{candidate_sets_str}')
""").to_pandas()

# Build requirements dictionary
requirements_dict = {}
for _, row in requirements_df.iterrows():
    set_num = row['SET_NUM']
    if set_num not in requirements_dict:
        requirements_dict[set_num] = {}
    requirements_dict[set_num][row['PART_COLOR_ID']] = row['QUANTITY']

print(f"Inventory: {len(inventory_dict):,} part-color combinations")
print(f"Candidates: {len(requirements_dict):,} sets")
print(f"Requirements matrix: {len(requirements_df):,} entries")

## Step 5: Run Large-Scale Optimization

In [ ]:
import time

print("Building large-scale optimization model...")
start_time = time.time()

# Create model
model = gp.Model("ToyBrickLargeScale")
model.setParam('OutputFlag', 1)  # Show output for large model
model.setParam('TimeLimit', 600)  # 10 minute time limit
model.setParam('MIPGap', 0.02)  # Accept 2% optimality gap

# Decision variables
set_vars = {}
for set_num in candidate_list:
    set_vars[set_num] = model.addVar(vtype=GRB.BINARY, name=f"build_{set_num}")

print(f"Created {len(set_vars):,} decision variables")

# Objective: Maximize parts used
objective_expr = gp.LinExpr()
for set_num in candidate_list:
    num_parts = candidate_sets_df[candidate_sets_df['SET_NUM'] == set_num]['NUM_PARTS'].values[0]
    objective_expr += num_parts * set_vars[set_num]

model.setObjective(objective_expr, GRB.MAXIMIZE)

# Constraints
print("Adding inventory constraints...")
constraint_count = 0
for part_color_id, available in inventory_dict.items():
    constraint_expr = gp.LinExpr()
    
    for set_num in candidate_list:
        if set_num in requirements_dict and part_color_id in requirements_dict[set_num]:
            required_qty = requirements_dict[set_num][part_color_id]
            constraint_expr += required_qty * set_vars[set_num]
    
    if constraint_expr.size() > 0:
        model.addConstr(constraint_expr <= available, name=f"inv_{part_color_id}")
        constraint_count += 1

print(f"Added {constraint_count:,} constraints")

model.update()
setup_time = time.time() - start_time

print(f"\nModel Statistics:")
print(f"  Variables: {model.NumVars:,}")
print(f"  Constraints: {model.NumConstrs:,}")
print(f"  Setup time: {setup_time:.2f} seconds")
print(f"\nSolving...")

In [ ]:
# Solve the model
solve_start = time.time()
model.optimize()
solve_time = time.time() - solve_start

print(f"\nSolve time: {solve_time:.2f} seconds")

## Step 6: Analyze Large-Scale Solution

In [ ]:
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT:
    print("\n" + "="*70)
    if model.status == GRB.OPTIMAL:
        print("OPTIMAL SOLUTION FOUND")
    else:
        print(f"BEST SOLUTION FOUND (within {model.MIPGap*100:.2f}% of optimal)")
    print("="*70)
    
    # Get selected sets
    selected_sets = []
    for set_num, var in set_vars.items():
        if var.X > 0.5:
            selected_sets.append(set_num)
    
    total_inventory = sum(inventory_dict.values())
    
    print(f"\nResults:")
    print(f"  Sets selected: {len(selected_sets):,}")
    print(f"  Total parts used: {model.ObjVal:,.0f}")
    print(f"  Total inventory: {total_inventory:,}")
    print(f"  Parts leftover: {total_inventory - model.ObjVal:,.0f}")
    print(f"  Utilization: {(model.ObjVal / total_inventory * 100):.1f}%")
    print(f"  Solve time: {solve_time:.1f} seconds")
    
    # Save solution to Snowflake
    solution_records = []
    for set_num in selected_sets:
        set_info = candidate_sets_df[candidate_sets_df['SET_NUM'] == set_num].iloc[0]
        solution_records.append({
            'SET_NUM': set_num,
            'SET_NAME': set_info['SET_NAME'],
            'YEAR': int(set_info['YEAR']),
            'THEME_NAME': set_info['THEME_NAME'],
            'NUM_PARTS': int(set_info['NUM_PARTS'])
        })
    
    solution_df = pd.DataFrame(solution_records)
    session.create_dataframe(solution_df).write.mode('overwrite').save_as_table(
        'optimization_solution_large'
    )
    
    print(f"\nSolution saved to table: optimization_solution_large")
    
    # Show summary by theme
    print("\nSelected sets by theme:")
    session.sql("""
    SELECT 
        theme_name,
        COUNT(*) as num_sets,
        SUM(num_parts) as total_parts,
        AVG(num_parts) as avg_parts
    FROM optimization_solution_large
    GROUP BY theme_name
    ORDER BY num_sets DESC
    """).show()
    
    # Show sample of selected sets
    print("\nSample of selected sets:")
    session.sql("""
    SELECT set_num, set_name, year, theme_name, num_parts
    FROM optimization_solution_large
    ORDER BY num_parts DESC
    LIMIT 15
    """).show()
    
else:
    print(f"Optimization failed with status: {model.status}")

## Step 7: Compare with Alternative Strategies

In [ ]:
# Compare optimization result with simple heuristics

print("Comparing optimization with heuristic strategies...\n")

# Heuristic 1: Greedy - pick largest sets first
greedy_sets = []
greedy_inventory = inventory_dict.copy()
greedy_parts = 0

for _, row in candidate_sets_df.sort_values('NUM_PARTS', ascending=False).iterrows():
    set_num = row['SET_NUM']
    
    if set_num not in requirements_dict:
        continue
    
    # Check if we can build this set
    can_build = True
    for part_color_id, qty_needed in requirements_dict[set_num].items():
        if greedy_inventory.get(part_color_id, 0) < qty_needed:
            can_build = False
            break
    
    if can_build:
        greedy_sets.append(set_num)
        greedy_parts += row['NUM_PARTS']
        # Deduct parts
        for part_color_id, qty_needed in requirements_dict[set_num].items():
            greedy_inventory[part_color_id] -= qty_needed

# Heuristic 2: Pick smallest sets (maximize count)
small_first_sets = []
small_first_inventory = inventory_dict.copy()
small_first_parts = 0

for _, row in candidate_sets_df.sort_values('NUM_PARTS', ascending=True).iterrows():
    set_num = row['SET_NUM']
    
    if set_num not in requirements_dict:
        continue
    
    can_build = True
    for part_color_id, qty_needed in requirements_dict[set_num].items():
        if small_first_inventory.get(part_color_id, 0) < qty_needed:
            can_build = False
            break
    
    if can_build:
        small_first_sets.append(set_num)
        small_first_parts += row['NUM_PARTS']
        for part_color_id, qty_needed in requirements_dict[set_num].items():
            small_first_inventory[part_color_id] -= qty_needed

# Summary comparison
total_inventory = sum(inventory_dict.values())

print("="*70)
print("STRATEGY COMPARISON")
print("="*70)
print(f"{'Strategy':<30} {'Sets':<10} {'Parts Used':<15} {'Utilization':<12}")
print("-"*70)

if model.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    opt_util = model.ObjVal / total_inventory * 100
    print(f"{'Gurobi Optimization':<30} {len(selected_sets):<10,} {model.ObjVal:<15,.0f} {opt_util:<12.2f}%")

greedy_util = greedy_parts / total_inventory * 100
print(f"{'Greedy (largest first)':<30} {len(greedy_sets):<10,} {greedy_parts:<15,} {greedy_util:<12.2f}%")

small_util = small_first_parts / total_inventory * 100
print(f"{'Small first (max count)':<30} {len(small_first_sets):<10,} {small_first_parts:<15,} {small_util:<12.2f}%")

print("="*70)

if model.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    improvement_vs_greedy = ((model.ObjVal - greedy_parts) / greedy_parts * 100)
    improvement_vs_small = ((model.ObjVal - small_first_parts) / small_first_parts * 100)
    
    print(f"\nOptimization improvement:")
    print(f"  vs Greedy: {improvement_vs_greedy:+.1f}% more parts used")
    print(f"  vs Small-first: {improvement_vs_small:+.1f}% more parts used")

## Step 8: Sensitivity Analysis

Analyze which constraints are binding and how the solution changes with different parameters.

In [ ]:
if model.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    print("Analyzing constraint sensitivity...\n")
    
    # Find binding constraints (fully utilized part-colors)
    binding_parts = []
    
    for part_color_id in inventory_dict.keys():
        total_used = 0
        for set_num in selected_sets:
            if set_num in requirements_dict and part_color_id in requirements_dict[set_num]:
                total_used += requirements_dict[set_num][part_color_id]
        
        available = inventory_dict[part_color_id]
        utilization = (total_used / available * 100) if available > 0 else 0
        
        if utilization >= 99:  # Essentially fully utilized
            binding_parts.append({
                'part_color_id': part_color_id,
                'available': available,
                'used': total_used,
                'utilization': utilization
            })
    
    print(f"Binding constraints (fully utilized parts): {len(binding_parts)}")
    print(f"Total part-color combinations: {len(inventory_dict)}")
    print(f"Percentage binding: {len(binding_parts) / len(inventory_dict) * 100:.1f}%")
    
    if binding_parts:
        print("\nTop 10 fully utilized part-colors:")
        binding_df = pd.DataFrame(binding_parts).sort_values('available', ascending=False).head(10)
        
        # Get part names
        part_ids = "','".join(binding_df['part_color_id'].tolist())
        part_names = session.sql(f"""
        SELECT part_color_id, part_name, color_name
        FROM part_color_pairs
        WHERE part_color_id IN ('{part_ids}')
        """).to_pandas()
        
        binding_df = binding_df.merge(part_names, on='part_color_id', how='left')
        print(binding_df[['part_name', 'color_name', 'available', 'used']].to_string(index=False))

## Summary

This notebook demonstrated:

1. **Large-Scale Problem Formulation**: Handled 100+ owned sets and 500+ candidate sets
2. **Performance Optimization**: Used appropriate Gurobi parameters for large models
3. **Comparison with Heuristics**: Showed value of optimization vs. simple greedy approaches
4. **Sensitivity Analysis**: Identified bottleneck parts that limit the solution
5. **Snowflake Integration**: Stored results in tables for further analysis

## Next Steps

- **Experiment with different objectives**: Value-based (prioritize expensive sets), diversity (spread across themes)
- **Add business constraints**: Budget limits, specific theme requirements, minimum set sizes
- **Multi-objective optimization**: Balance multiple goals simultaneously
- **What-if analysis**: How does solution change with 10% more inventory? Different owned sets?
- **Visualization**: Create dashboards showing selected sets, utilization, and comparisons